# Ch 26 — 도메인 요약·생성 (Decoder LoRA + 추가 사전학습)

Continued pre-training (CPT) 과 Domain SFT (LoRA) 두 단계 도메인 적응. Qwen 2.5-0.5B-Instruct 에 한국 동화 instruction 페어 LoRA. 평가 + 어댑터 합치기 + GGUF 변환 다리.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part7/ch26_domain_lora.ipynb)

In [ ]:
# !pip install -q transformers peft datasets accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Continued pre-training (CPT) — 언제 필요한가

| 상황 | CPT 필요 | 이유 |
|---|---|---|
| 베이스가 도메인 어휘 모름 (의료, 법률) | 필요 | 어휘 확장 |
| 베이스가 한국어 일반은 OK, 도메인은 약함 | 상황따라 | 도메인 raw 1B+ 있으면 |
| 베이스가 도메인 일반 OK, 형식만 맞추기 | 불필요 | LoRA 만 |
| 본 책 캡스톤 (한국어 동화) | 불필요 | Qwen 2.5 한국어 충분 |

CPT 를 하려면 raw 도메인 text **최소 100M 토큰**.

In [ ]:
# cpt.py — CPT 하는 법 (선택, 도메인 raw text 있을 때)
# 실제 실행 시: base 모델 (instruct 아님) + raw text corpus 필요

# from transformers import Trainer, TrainingArguments, AutoModelForCausalLM
# from peft import LoraConfig, get_peft_model

# base = "Qwen/Qwen2.5-0.5B"          # base 모델 (instruct 아님)
# model = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.bfloat16)

# lora_cfg = LoraConfig(r=64, lora_alpha=128,      # CPT 는 r 크게
#                        target_modules=["q_proj","k_proj","v_proj","o_proj",
#                                         "gate_proj","up_proj","down_proj"],  # FFN 도
#                        task_type="CAUSAL_LM")
# model = get_peft_model(model, lora_cfg)

# 데이터: raw text concat
# ds = load_dataset("text", data_files="domain_corpus.txt")["train"]
# ds = ds.map(lambda x: tok(x["text"], max_length=1024, truncation=True), batched=True)

# trainer = Trainer(model=model, args=TrainingArguments(
#     output_dir="cpt_out", num_train_epochs=1,        # CPT 는 보통 1 epoch
#     learning_rate=2e-4, per_device_train_batch_size=8,
#     bf16=True, save_steps=500), train_dataset=ds)
# trainer.train()

print("CPT 코드 (주석). domain_corpus.txt + base 모델이 있을 때 실행.")
print("CPT 의 핵심: gate/up/down_proj (FFN) 도 target_modules 에 포함.")

## 3. Domain SFT (LoRA) — 본 책 캡스톤 길 (최소 예제)

In [ ]:
# domain_lora.py
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import torch, json, os

base = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.bfloat16, device_map="auto")

lora = LoraConfig(r=16, lora_alpha=32,
                   target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                   lora_dropout=0.05, task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 4. 본 책 캡스톤 — 한국 동화 데이터 페어 (실전)

In [ ]:
# story_pairs.py — 합성 동화에 instruction 형식 부여
import json, os, random

TEMPLATES = [
    "3~5세 어린이용 한국어 동화 한 편을 만들어줘. 따뜻한 톤으로.",
    "{character}가 등장하는 짧은 동화를 써줘.",
    "{keyword}에 대한 동화를 200자로 만들어줘.",
    "어린이가 잠들기 전 듣는 따뜻한 동화 한 편.",
]
CHARACTERS = ["토끼 토토", "곰 두두", "할머니", "고양이 미미"]
KEYWORDS = ["당근", "비", "달", "친구", "엄마", "꽃"]

SAMPLE_STORIES = [
    "옛날 옛날에 작은 토끼가 살았어요. 토끼는 매일 아침 당근을 먹었지요. 어느 날, 토끼는 친구 다람쥐를 만났어요. 둘은 사이좋게 당근을 나눠 먹었답니다.",
    "숲속에 작은 곰이 살았어요. 곰은 꿀을 찾아 매일 산을 올랐어요. 비가 오는 날에도 곰은 포기하지 않았지요. 마침내 달콤한 꿀을 찾아 친구들과 나눴어요.",
    "달밤에 작은 고양이가 창문 밖을 바라봤어요. 엄마가 이야기해줬어요. 달님도 잠을 자야 한대요. 고양이도 엄마 품에 안겨 스르르 잠들었답니다.",
]

pairs = []
for i in range(30):  # 더미 30 페어 (실제로는 5K~50K)
    tmpl = random.choice(TEMPLATES)
    instruction = tmpl
    if "{character}" in tmpl:
        instruction = tmpl.format(character=random.choice(CHARACTERS))
    elif "{keyword}" in tmpl:
        instruction = tmpl.format(keyword=random.choice(KEYWORDS))
    pairs.append({
        "instruction": instruction,
        "output": random.choice(SAMPLE_STORIES),
    })

with open("domain_pairs.jsonl", "w") as f:
    for p in pairs: f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"동화 페어 {len(pairs)} 개 저장")

In [ ]:
# 데이터 포맷 + 학습
def fmt(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    text = tok.apply_chat_template(msgs, tokenize=False)
    enc = tok(text, max_length=512, truncation=True, padding="max_length")
    enc["labels"] = enc["input_ids"].copy()
    return enc

ds = load_dataset("json", data_files="domain_pairs.jsonl")["train"]
ds = ds.map(fmt).remove_columns(["instruction", "output"])

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="lora_out", num_train_epochs=3, learning_rate=1e-4,
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        warmup_steps=20, lr_scheduler_type="cosine",
        bf16=(device == "cuda"), save_steps=200, logging_steps=10,
    ),
    train_dataset=ds, tokenizer=tok
)
trainer.train()
model.save_pretrained("lora_out/adapter")
print("어댑터 저장 완료: lora_out/adapter/")

## 5. 평가 — Part 5 응용

In [ ]:
# eval_lora.py — 베이스 vs LoRA 비교
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base_id = "Qwen/Qwen2.5-0.5B-Instruct"
tok_eval = AutoTokenizer.from_pretrained(base_id)
base_model = AutoModelForCausalLM.from_pretrained(base_id, torch_dtype=torch.bfloat16, device_map="auto")
lora_model = PeftModel.from_pretrained(base_model, "lora_out/adapter")
lora_model.eval()

def generate_story(model, prompt, max_new=300):
    msgs = [{"role": "user", "content": prompt}]
    ids = tok_eval.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True)
    ids = ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, temperature=0.8, top_p=0.9, do_sample=True)
    return tok_eval.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

prompt = "3~5세 어린이용 한국어 동화 한 편을 만들어줘. 따뜻한 톤으로."
print("=== LoRA 모델 출력 ===")
print(generate_story(lora_model, prompt))

## 6. 어댑터 합치기 + GGUF 변환

In [ ]:
# merge_export.py
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base_id = "Qwen/Qwen2.5-0.5B-Instruct"
tok_merge = AutoTokenizer.from_pretrained(base_id)
base = AutoModelForCausalLM.from_pretrained(base_id, torch_dtype=torch.bfloat16)
model_merge = PeftModel.from_pretrained(base, "lora_out/adapter")
merged = model_merge.merge_and_unload()   # base + adapter 합치기
merged.save_pretrained("merged_model")
tok_merge.save_pretrained("merged_model")
print("합치기 완료: merged_model/")

# GGUF 변환 (llama.cpp 필요)
# !python llama.cpp/convert_hf_to_gguf.py merged_model \
#     --outfile dist/tiny-tale-ko.gguf --outtype f16
# !./llama.cpp/llama-quantize \
#     dist/tiny-tale-ko.gguf dist/tiny-tale-ko-q4km.gguf Q4_K_M
print("GGUF 변환 명령 (주석 해제 후 실행):")
print("  python llama.cpp/convert_hf_to_gguf.py merged_model --outfile dist/tiny-tale-ko.gguf")

## 연습문제

1. Qwen 2.5-0.5B-Instruct 에 본인 도메인 페어 1,000 LoRA. 학습 전·후 PPL 차이?
2. **베이스 능력 회귀** — 일반 한국어 prompt 10개에 베이스 vs LoRA 답변 비교. 어느 쪽이 더 자연스러운가?
3. r=8 / 16 / 32 의 학습 결과 비교. 어디서 sweet spot?
4. CPT (raw 동화 100K) → SFT (페어) vs SFT only 비교. CPT 효과는?
5. **(생각해볼 것)** "본 책 10M from-scratch" 와 "Qwen 0.5B + LoRA" — 같은 동화 도메인에서 어느 쪽이 나은가? 어느 측면에서 다를까?

In [ ]:
# 연습 1: 본인 도메인 페어 1,000 LoRA + PPL


In [ ]:
# 연습 2: 베이스 능력 회귀 검증
general_prompts = [
    "오늘 날씨가 좋네요. 뭘 하면 좋을까요?",
    "파이썬에서 리스트 정렬하는 방법은?",
    # 추가 prompt
]
# 베이스 vs LoRA 비교 코드 작성


In [ ]:
# 연습 3: r=8 / 16 / 32 비교
